In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# standardised type-token ration (type=lemma)

import pandas as pd
import numpy as np

# =========================
# USER SETTINGS
# =========================
FILE_PATH = "/content/drive/MyDrive/FLO/segments_dataset.xlsx"
TEXT_COL = "transcript"

WINDOW_SIZE = 10

df = pd.read_excel(FILE_PATH, dtype=str)

def compute_sttr(tokens, window_size=10):
    tokens = [t for t in tokens if t.strip() != ""]
    n_tokens = len(tokens)

    if n_tokens == 0:
        return np.nan

    if n_tokens < window_size:
        return len(set(tokens)) / n_tokens

    ttr_values = []

    for i in range(n_tokens - window_size + 1):
        window = tokens[i:i + window_size]
        ttr = len(set(window)) / window_size
        ttr_values.append(ttr)

    return np.mean(ttr_values)

sttr_results = []

for text in df[TEXT_COL]:
    if pd.isna(text):
        sttr_results.append(np.nan)
    else:
        tokens = text.split()
        sttr_results.append(compute_sttr(tokens, WINDOW_SIZE))

df["STTR"] = sttr_results

# ----------------------------
# SAVE OUTPUT
# ----------------------------
OUTPUT_PATH = "/content/drive/MyDrive/FLO/segments_dataset_with_STTR.xlsx"
df.to_excel(OUTPUT_PATH, index=False)

print("STTR computation completed successfully.")

In [ ]:
#content complexity (mean ortographic wordlength)

import pandas as pd
import numpy as np

# ----------------------------
# USER SETTINGS
# ----------------------------
FILE_PATH = "/content/drive/MyDrive/FLO/segments_dataset.xlsx"
TEXT_COL = "transcript"


df = pd.read_excel(FILE_PATH, dtype=str)


def mean_orthographic_word_length(tokens):
    tokens = [t for t in tokens if t.strip() != ""]
    if len(tokens) == 0:
        return np.nan
    return np.mean([len(t) for t in tokens])


complexity_values = []

for text in df[TEXT_COL]:
    if pd.isna(text):
        complexity_values.append(np.nan)
    else:
        tokens = text.split()
        complexity_values.append(mean_orthographic_word_length(tokens))

df["content_complexity_mean_orthographic_word_length"] = complexity_values

# ----------------------------
# SAVE OUTPUT
# ----------------------------
OUTPUT_PATH = "/content/drive/MyDrive/FLO/segments_dataset_with_content_complexity.xlsx"
df.to_excel(OUTPUT_PATH, index=False)

print("Content complexity computation completed successfully.")

In [ ]:
#retrieving English discourse markers#

import re
import pandas as pd
import spacy
nlp = spacy.load("en_core_web_sm")

# ----------------------------
# USER SETTINGS
# ----------------------------
FILE_PATH = "/content/drive/MyDrive/FLO/segments_dataset.xlsx"
TEXT_COL = "transcript"

patterns = {
    "I mean": re.compile(r'\bI mean\b', re.IGNORECASE),  # Matches 'I mean'
    "Actually": re.compile(r'\bActually\b', re.IGNORECASE),  # Matches 'Actually' anywhere
    "Like": re.compile(r'\blike\b(?!\s(to|a|\b\w+ing\b))', re.IGNORECASE),  # Excludes verb/preposition use
    "You know": re.compile(r'(?<!\bdo\s)\byou know\b', re.IGNORECASE),  # Excludes 'do you know'
    "Yeah (filler)": re.compile(r'\bYeah\b', re.IGNORECASE)  # Matches all "yeah" initially
}

# Function to count occurrences of 'So' NOT preceding an adjective and 'Well' NOT following a verb
def count_filtered_words_spacy(text, word_type):
    doc = nlp(text)
    count = 0

    for i, token in enumerate(doc):
        if word_type == "So" and token.text.lower() == "so":
            # Exclude if followed by an adjective
            if i < len(doc) - 1 and doc[i + 1].pos_ == "ADJ":
                continue
            count += 1

        elif word_type == "Well" and token.text.lower() == "well":
            # Exclude if preceded by a verb
            if i > 0 and doc[i - 1].pos_ == "VERB":
                continue
            count += 1

    return count

# Apply regex patterns to count occurrences
for key, pattern in patterns.items():
    df[key] = df[TEXT_COL].astype(str).apply(lambda x: len(pattern.findall(str(x))))

# Apply functions to count occurrences of 'So' and 'Well' with POS filtering
df["So"] = df[TEXT_COL].astype(str).apply(lambda x: count_filtered_words_spacy(x, "So"))
df["Well"] = df[TEXT_COL].astype(str).apply(lambda x: count_filtered_words_spacy(x, "Well"))

# Function to count 'Really' while excluding adjectives, turn-initial position, and questions
def count_really(text):
    doc = nlp(text)
    count = 0
    for i, token in enumerate(doc):
        if token.text.lower() == "really":
            # Excludes 'Really' before adjectives
            if i < len(doc) - 1 and doc[i + 1].pos_ == "ADJ":
                continue  # Skips 'Really' if modifying an adjective
            # Excludes 'Really' at the beginning of a turn
            if i == 0:
                continue  # Skips "Really" at turn-initial position
            # Excludes "Really" before a question mark
            if i < len(doc) - 1 and doc[i + 1].text == "?":
                continue # Skips "Really" if followed by a question mark
            count += 1 # Only count if none of the exclusion conditions are met
    return count

df["Really"] = df[TEXT_COL].astype(str).apply(count_really)

# ----------------------------
# SAVE OUTPUT
# ----------------------------
OUTPUT_PATH = "/content/drive/MyDrive/FLO/segments_dataset_with_discourse_markers.xlsx"
df.to_excel(OUTPUT_PATH, index=False)

print("Discourse marker computation completed successfully.")

In [ ]:
#Calculating repetitions and false starts#
#This code captures the following cases of repetitions: 1) words that are immediately repeated 2) words repeated in a span of 3 tokens 3) 2grams and 3grams that are repeated in a span of 3 tokens.

import re
import pandas as pd

# ----------------------------
# USER SETTINGS
# ----------------------------
FILE_PATH = "/content/drive/MyDrive/FLO/segments_dataset.xlsx"
TEXT_COL = "transcript"

df = pd.read_excel(FILE_PATH)

repetition_patterns = {
    "Rep1": re.compile(r'\b(\w+)\s+\1\b'),  # Rep1
    "Rep2": re.compile(r'\b(\w+)\b(?:\s+\b(?!\1)\w+\b){0,2}?\s+\1\b'),  # Rep2
    "Rep3": re.compile(r'\b(\w+\s+\w+|\w+\s+\w+\s+\w+)\b(?:\s+\b(?!\1)\w+\b){0,2}?\s+\1\b'),  # Rep3
}
for col_name in ["Rep1", "Rep2", "Rep3"]:
    df[col_name] = 0
for key, pattern in repetition_patterns.items():
    df[key] = df[TEXT_COL].astype(str).apply(lambda x: len(pattern.findall(x)))
OUTPUT_PATH = "/content/drive/MyDrive/FLO/segments_dataset_with_repetitions.xlsx"
df.to_excel(OUTPUT_PATH, index=False)


